In [14]:
import logging
from groq import RateLimitError, APIConnectionError, APITimeoutError
from pydantic import ValidationError
import os
import time
from groq import Groq
from pydantic import BaseModel
from datetime import datetime
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)
class Advice(BaseModel):
    answer: str
    topic: str
    difficulty: str
class Calculate(BaseModel):
    a: float
    b: float
    operation_type: str
history = []
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)
def calculate(a:float,b:float, operation_type:str ):
    operation_type = operation_type.strip().lower()
    if operation_type == "+":
        return a+b
    elif operation_type == "-":
        return a-b
    elif operation_type == "*":
        return a*b
    elif operation_type == "/" and b != 0:
        return a/b
    else:
        raise ValueError(f"Unsupported operation: {operation_type}")
def get_time():
    return datetime.now()
TOOL_REGISTRY = {
    "calculate": {
        "function": calculate,
        "schema": Calculate,
    },
    "get_time": {
        "function": get_time,
        "schema": None
    }
}
ALLOWED_TOOLS = {"calculate","get_time"}
def execute_tool(tool_call):
    tool_name = tool_call.function.name
    tool_arguments = tool_call.function.arguments
    if tool_name not in ALLOWED_TOOLS:
        raise PermissionError(f"Tool '{tool_name}' is not allowed.")
    tool = TOOL_REGISTRY.get(tool_name)
    if tool is None:
        raise ValueError(f"Unknown tool: {tool_name}")
    try:
        if tool["schema"] != None:
            args = tool["schema"].model_validate_json(tool_arguments)
            return tool["function"](**args.model_dump())
        else:
            return tool["function"]()
    except ValidationError as e:
        return f"Tool validation failed: {e}"
system_prompt = {
    "role":"system",
    "content": """You are a tutor for school-going kids. Your answers should be short and easy to understand.
        Return your response ONLY as JSON with exactly these fields:
        answer: string
        topic: string
        difficulty: string
        When a calculation or current date/time is needed, ALWAYS use the provided tools.
        IMPORTANT:
        - Never write <function=...> in your response.
        - Never write tool calls as text.
        - Never simulate or describe a tool call.
        - Use the actual tool calling mechanism provided by the API.
        - After receiving a tool result, continue using tools if another calculation is needed.
        - Only produce the final JSON after all required tools are completed.
        Example:
        {
            "answer": "Gravity is a force that pulls objects toward each other.",
            "topic": "Physics",
            "difficulty": "Easy"
        }
    """
}
calculate_tool = {
    "type":"function",
    "function": {
        "name":"calculate",
        "description": """Performs basic arithmetic using two numbers and one operation. Supported operations are
            addition (+), subtraction (-), multiplication (*), and division (/). Division by zero is not allowed.""",
        "parameters": {
            "type":"object",
            "properties": {
                "a":{
                    "type":"number"
                },
                "b":{
                    "type":"number"
                },
                "operation_type":{
                    "type":"string",
                    "enum":["+","-","*","/"]
                }
            },
            "required":["a","b","operation_type"]
        }
    }
}
get_current_time_tool = {
    "type":"function",
    "function": {
        "name":"get_time",
        "description": """this function returns current time""",
        "parameters": {
            "type":"object",
            "properties": {},
            "required":[]
        }
    }
}
history.append(system_prompt)
def CREATE_CONNECTION_WITH_GROQ(history, tools, stream):
    return client.chat.completions.create(
        messages = history,
        model="llama-3.3-70b-versatile",
        tools=tools,
        stream=stream,
        tool_choice="auto",
        temperature=0.2
    )
ALL_TOOLS_LIST = [calculate_tool, get_current_time_tool]
def streamIt(chat_completion):
    ai_ans=""
    for chunk in chat_completion:
        is_present = chunk.choices[0].delta.content == None
        if is_present == False:
            print(chunk.choices[0].delta.content, end="", flush=True)
            ai_ans += chunk.choices[0].delta.content
    return ai_ans
while(True):
    user_input = input("Enter your query ...")
    if user_input.strip() == "":
        print("Please enter a valid question...")
        continue
    if ["stop", "exit", "break", "quit"].__contains__(user_input.strip().lower()):
        break
    elif user_input.strip().lower() == "reset":
        history = []
        history.append(system_prompt)
        continue
    history.append({
        "role": "user",
        "content": user_input
    })
    try:   
        chat_completion = CREATE_CONNECTION_WITH_GROQ(history, ALL_TOOLS_LIST, False)
        message = chat_completion.choices[0].message
        history.append(message)
        if message.tool_calls:
            while message.tool_calls:
                for toolCall in range(len(message.tool_calls)):
                    result = execute_tool(message.tool_calls[toolCall])
                    history.append({
                        "role":"tool",
                        "tool_call_id": message.tool_calls[toolCall].id,
                        "content" : str(result)
                    })
                tool_final_response = CREATE_CONNECTION_WITH_GROQ(history, ALL_TOOLS_LIST, False)
                message = tool_final_response.choices[0].message
                history.append(message)
        if not message.tool_calls:
            try:
                ai_ans = message.content
                print(ai_ans, end="", flush=True)
                try:
                    advice = Advice.model_validate_json(ai_ans)                   
                except ValidationError:
                    print("\nAI response is not following expected model structure.")
            except Exception as e:
                logger.info(f"Final response error: {e}")
    except KeyboardInterrupt as e:
        logger.info("Chatbot stopped...")
        break
    except APIConnectionError as e:
        logger.error("Connection can't be established.")
    except RateLimitError as e:
        logger.error("Too many requests. Please wait...")
    except APITimeoutError as e:
        logger.error("It is taking too long...")
    except Exception as e:

        logger.error(f"Error{e}")

Personal terminal based Agent. It has access to multiple tools.
It can ans questions like
what is gravity?
what is 10 multiplied by 10?
what is the current time?
multiply 10 with 10 and add the no of day of this month.